In [0]:
from delta.tables import DeltaTable
import os
import sys

# Ensure Python can find utils.py from the parent directory
project_dir = "/Workspace/Users/ranayuvrajsingh1111@gmail.com/Olist ecomerce ETL"
if project_dir not in sys.path:
  sys.path.append(project_dir)

import utils
from pyspark.sql import functions as F

# ==========================================
# 1. PARAMETERIZATION (Databricks Widget)
# ==========================================
dbutils.widgets.text("batch_name", "batch_1", "Target Batch Name")
batch_name = dbutils.widgets.get("batch_name")

print(f"Processing Silver Order Items for Batch: {batch_name}")

# ==========================================
# 2. EXTRACT INCOMING DATA & REFERENCE TABLES
# ==========================================
df_items_raw = spark.read.table(
    "ecommerce_lakehouse.bronze.bronze_olist_order_items"
).filter(F.col("_batch_id") == batch_name)

df_products = spark.read.table("ecommerce_lakehouse.silver.silver_products")
df_sellers = spark.read.table("ecommerce_lakehouse.silver.silver_sellers")

# ==========================================
# 3. TRANSFORMATIONS & DIMENSION MAPPING
# ==========================================
# Clean base attributes and cast types
df_cleaned = df_items_raw.select(
    F.trim(F.col("order_id")).alias("order_id"),
    F.col("order_item_id").cast("int").alias("order_item_id"),
    F.trim(F.col("product_id")).alias("product_id"),
    F.trim(F.col("seller_id")).alias("seller_id"),
    F.col("shipping_limit_date").cast("timestamp").alias("shipping_limit_ts"),
    F.coalesce(F.col("price").cast("double"), F.lit(0.0)).alias("price"),
    F.coalesce(F.col("freight_value").cast("double"), F.lit(0.0)).alias(
        "freight_value"
    ),
).dropDuplicates(["order_id", "order_item_id"])

# Map product_id to product_key
df_with_product = df_cleaned.join(
    df_products.select("product_id", "product_key"), on="product_id", how="left"
).withColumn(
    "product_key",
    F.coalesce(F.col("product_key"), F.lit("P_UNKNOWN")),
)

# Map seller_id to seller_key
df_with_keys = df_with_product.join(
    df_sellers.select("seller_id", "seller_key"), on="seller_id", how="left"
).withColumn(
    "seller_key",
    F.coalesce(F.col("seller_key"), F.lit("S_UNKNOWN")),
)

# Compute Row Hash using SHA-256 for change detection
df_hashed = df_with_keys.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("product_key"), F.lit("")),
            F.coalesce(F.col("seller_key"), F.lit("")),
            F.coalesce(F.col("shipping_limit_ts").cast("string"), F.lit("")),
            F.coalesce(F.col("price").cast("string"), F.lit("")),
            F.coalesce(F.col("freight_value").cast("string"), F.lit("")),
        ),
        256,
    ),
)

# ==========================================
# 4. TARGET TABLE INITIALIZATION OR UPSERT
# ==========================================
target_table_name = "ecommerce_lakehouse.silver.silver_order_items"
table_exists = spark.catalog.tableExists(target_table_name)
current_timestamp = F.current_timestamp()

if not table_exists:
  print("Target table does not exist. Initializing silver_order_items...")

  df_transformed = df_hashed.repartition(1).select(
      "order_id",
      "order_item_id",
      "product_id",
      "product_key",
      "seller_id",
      "seller_key",
      "shipping_limit_ts",
      "price",
      "freight_value",
      "row_hash",
      current_timestamp.alias("_etl_inserted_at"),
      current_timestamp.alias("_etl_updated_at"),
      F.lit(False).alias("_delete_flag"),
  )

  df_transformed.write.format("delta").mode("overwrite").saveAsTable(
      target_table_name
  )
  print("Successfully initialized silver_order_items table!")

else:
  print("Table exists. Executing incremental upsert merge...")

  deltaTable = DeltaTable.forName(spark, target_table_name)

  deltaTable.alias("target").merge(
      source=df_hashed.alias("source"),
      condition=(
          "target.order_id = source.order_id AND target.order_item_id ="
          " source.order_item_id"
      ),
  ).whenMatchedUpdate(
      condition="target.row_hash <> source.row_hash",
      set={
          "product_id": F.col("source.product_id"),
          "product_key": F.col("source.product_key"),
          "seller_id": F.col("source.seller_id"),
          "seller_key": F.col("source.seller_key"),
          "shipping_limit_ts": F.col("source.shipping_limit_ts"),
          "price": F.col("source.price"),
          "freight_value": F.col("source.freight_value"),
          "row_hash": F.col("source.row_hash"),
          "_etl_updated_at": current_timestamp,
      },
  ).whenNotMatchedInsert(
      values={
          "order_id": F.col("source.order_id"),
          "order_item_id": F.col("source.order_item_id"),
          "product_id": F.col("source.product_id"),
          "product_key": F.col("source.product_key"),
          "seller_id": F.col("source.seller_id"),
          "seller_key": F.col("source.seller_key"),
          "shipping_limit_ts": F.col("source.shipping_limit_ts"),
          "price": F.col("source.price"),
          "freight_value": F.col("source.freight_value"),
          "row_hash": F.col("source.row_hash"),
          "_etl_inserted_at": current_timestamp,
          "_etl_updated_at": current_timestamp,
          "_delete_flag": F.lit(False),
      },
  ).execute()

  print("Silver Order Items processing completed successfully!")

Processing Silver Order Items for Batch: batch_1
Target table does not exist. Initializing silver_order_items...
Successfully initialized silver_order_items table!
